# Dart board optimisation via exhaustion

In [1]:
#Dart board measurements in mm
BOARD_DIAMETER = 451
BULSEYE_INNER_DIAMETER = 12.7
BULSEYE_OUTER_DIAMETER = 32
CENTER_TO_OUTSIDE_DOUBLE = 170
CENTER_TO_OUTSIDE_TREBLE = 107
RING_SIZE = 8
#Normalised measurements (BOARD_DIAMETER = 1)
NORMALISED_BOARD_DIAMETER = 1
NORMALISED_BULSEYE_INNER_DIAMETER = 12.7/451
NORMALISED_BULSEYE_OUTER_DIAMETER = 32/451
NORMALISED_CENTER_TO_OUTSIDE_DOUBLE = 170/451
NORMALISED_CENTER_TO_OUTSIDE_TREBLE = 107/451
NORMALISED_RING_SIZE = 8/451

In [2]:
#Imports
import numpy as np
import math
import random
from abc import ABC, abstractmethod

In [3]:
#Array containg scores for the dart boards edges (Starting from 1)
SCORES = [1,18,4,13,6,10,15,2,17,3,19,7,16,8,11,14,9,12,5,20]

In [4]:
#Defining some helper functions
#Random 2 dp number between -1 and 1
def pnr2dp():
    return round((2*random.random()-1),2)
#Compute the angle 
def angle_between_vectors(u, v):
    """
    Calculate the angle between two vectors using the dot product.

    Parameters:
    u, v : array-like
        Input vectors.

    Returns:
    tuple
        The angle between the vectors in radians.
    
    Source:
    https://www.geeksforgeeks.org/python/how-to-compute-the-angle-between-vectors-using-python/
    """
    
    u = np.array(u)
    v = np.array(v)
    cos_theta = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    angle_rad = np.arccos(np.clip(cos_theta, -1.0, 1.0))
    return angle_rad

def calculate_score(x,y):
    """
    Calculate the resulting score from a hit on the board

    Parameters:
    x : number
        normalised x-coordinate of the hit
    y : number
        normalised x-coordinate of the hit
    """
    if x**2 + y**2 > (NORMALISED_CENTER_TO_OUTSIDE_DOUBLE**2)*4:
        return 0
    elif x**2 + y**2 < NORMALISED_BULSEYE_INNER_DIAMETER**2/4:
        return 50
    elif x**2 + y**2 < NORMALISED_BULSEYE_INNER_DIAMETER**2/4:
        return 25
    else:
        fractAround,_ = math.modf(1+angle_between_vectors([0,1],[x,y])/(2*math.pi)+1/40)
        base_score = math.floor(20*fractAround)%20
        return SCORES[base_score-1]

xx = pnr2dp()
yy = pnr2dp()
print(xx,yy,calculate_score(xx,yy))
xx = pnr2dp()
yy = pnr2dp()
print(xx,yy,calculate_score(xx,yy))
xx = pnr2dp()
yy = pnr2dp()
print(xx,yy,calculate_score(xx,yy))

-0.55 -0.29 15
-0.04 0.36 20
-0.76 -0.34 0


## Base class
We have a base scoring class that scores an aiming class

In [5]:
class Scoring_Object(ABC):
    @abstractmethod
    def scoring_function():
        pass

## Uniform circular accuracy
We compute the average score assuming that the players throws land in a uniform distribution in a circle around where they aim. We use a grid of points to estimate it to avoid difficult circle overlap calculations.

In [13]:
class Uniform_Circular(Scoring_Object):
    def scoring_function(self,r,cx,cy,fine):
        """
        Calculate the average score using a grid of sample points.
        Parameters:
        r : number
            radius or error of a throw
        cx : number 
            aimed x coord
        cy : number
            aimed y coord
        fine : number
            number of grid points in the square
        """
        count = 0
        total = 0
        for gx in range(fine+1):
            for gy in range(fine+1):
                xx = cx - r + (2*r*gx/fine)
                yy = cy - r + (2*r*gy/fine)
                if (xx - cx)**2 + (yy - cy)**2 < r**2:
                    count += 1
                    total += calculate_score(xx,yy)
        if count > 0:
            return total/count
        else:
            return 0


## Exhaustive search
Our first optimisation method is a brute force optimisation.

In [24]:
def exhaustive_optimisation(scorer,r,fine_1,fine_2):
    bestscore = -math.inf
    bestx = np.nan
    besty = np.nan
    for gx in range(fine_1+1):
        for gy in range(fine_1+1):
            xx = 2*(gx/fine_1)-1
            yy = 2*(gy/fine_1)-1
            a_score = scorer.scoring_function(r,xx,yy,fine_2)
            if (a_score > bestscore):
                bestscore = a_score
                bestx = xx
                besty = yy
    return [bestscore,bestx,besty]
uniscorer = Uniform_Circular()
print(exhaustive_optimisation(uniscorer,0.04,10,10))

[20.0, 0.0, 0.3999999999999999]
